In [0]:
import torch
from torch import nn

torch.__version__


# Data Preperation & Loading

In [0]:
# Create some data for a linear regression problem
weight = 0.7
bias = 0.3

start = 0
end = 1
step = 0.02

data_x = torch.arange(start, end, step).unsqueeze(dim=1)
label_y = weight * data_x + bias

data_x[:10], label_y[:10]


## Split Data into Training and Test Sets

Build a model that can learn the relationship between X (features) and y (labels)

| Split | Purpose | % of Total | Needed |
|-|-|-|-|
| Training Set   | The model learns from this data | ~60-80 | Always |
| Validation Set | The model is tuned from this data  | ~60-80 | Often |
| Testing Set    | The model is evaluated on this data to test what it has learned | ~60-80 | Always |

In this example we'll omit data validation.

In [0]:
# create a train / test split

train_split = int(0.8 * len(data_x))
x_train, y_train = data_x[:train_split], label_y[:train_split]
x_test, y_test = data_x[train_split:], label_y[train_split:]

len(x_train), len(y_train), len(x_test), len(y_test)

In [0]:
# visualise the data
import matplotlib.pyplot as plt

def plot_predictions(train_data,
                     train_labels,
                     test_data,
                     test_labels,
                     predictions=None):
    """
    Plots training data, test data and compares predictions.
    """
    plt.figure(figsize=(10, 7))
    # Plot training data in blue
    plt.scatter(train_data, train_labels, c="b", s=4, label="Training data")
    # Plot test data in green
    plt.scatter(test_data, test_labels, c="g", s=4, label="Testing data")
    if predictions is not None:
        # Plot the predictions in red (predictions were made on the test data)
        plt.scatter(test_data, predictions, c="r", s=4, label="Predictions")
    # Show the legend
    plt.legend(prop={"size": 14})
    plt.show()

plot_predictions(
    train_data=x_train,
    train_labels=y_train,
    test_data=x_test,
    test_labels=y_test
)

##  Build the Model

Standard linear regression model using PyTorch


In [0]:
class LinearRegressionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.weights = nn.Parameter(
            torch.randn(1, requires_grad=True, dtype=torch.float)
        )
        self.bias = nn.Parameter(
          torch.randn(1, requires_grad=True, dtype=torch.float)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
      # linear formula y = mx + b
      return self.weights * x + self.bias

In [0]:
# model parameters are initialised randomly, set the seed so our results are reproducible for learning purposes
torch.manual_seed(42)
model_0 = LinearRegressionModel()

# checkout the parameters intialised in the model
dict(model_0.state_dict())


In [0]:
# move the model to the GPU if available
device = "cpu"
if torch.cuda.is_available():
  device = "cuda"
  model_0.to(device)

elif torch.mps.is_available():
  device = "mps" 
  model_0.to(device) 

next(model_0.parameters()).device

In [0]:
# currently the model will perform very bad. Lets check it

with torch.inference_mode():
    y_preds = model_0(x_test)
# Check the predictions
print(f"Number of testing samples: {len(x_test)}") 
print(f"Number of predictions made: {len(y_preds)}")
print(f"Predicted values:\n{y_preds}")

In [0]:
plot_predictions(
    train_data=x_train,
    train_labels=y_train,
    test_data=x_test,
    test_labels=y_test,
    predictions=y_preds
)

# Train the Model

Iterate the model through gradient descent and back propagation to reduce the lost function by optimising the weights and bias. To do this we need to add a loss function and an optimiser.

| Function | What does it do? | Where in pytorch? | Common Values |
|-|-|-|-|
| Loss function | Measures how wrong predictions are compared to labels | `torch.nn` implements many built in loss functions | `L1Loss()` MAE </br>`BCELoss()` Binary Cross Entropy |
| Optimiser     | Tells model how to update it's internal parameters to reduce the loss function | `torch.optim` implements various optimisers | `SDG()` stochcastic gradient descent</br>`ADAM()` adam optimiser |



## pyTorch training loop

For the training loop, we'll build the following steps:
| Number | Step name | What does it do? | Code example |
|-|-|-|-|
|1 	| Forward pass | The model goes through all of the training data once, performing its forward() function calculations.| `model(x_train)` |
|2 	| Calculate the loss | The model's outputs (predictions) are compared to the ground truth and evaluated to see how wrong they are.|`loss = loss_fn(y_pred, y_train)`|
|3 	| Zero gradients | The optimizers gradients are set to zero (they are accumulated by default) so they can be recalculated for the specific training step.|`optimizer.zero_grad()`|
|4 	| Perform backpropagation on the loss | Computes the gradient of the loss with respect for every model parameter to be updated (each parameter with `requires_grad=True`). This is known as backpropagation, hence "backwards".|`loss.backward()`|
|5 	| Update the optimizer (gradient descent)|Update the parameters with requires_grad=True with respect to the loss gradients in order to improve them.|`optimizer.step()`|

## PyTorch testing loop

As for the testing loop (evaluating our model), the typical steps include:
|Number|Step name|What does it do?|Code example|
|-|-|-|-|
|1 | Forward pass | The model goes through all of the testing data once, performing its `forward()` function calculations. | `model(x_test)`|
|2 | Calculate the loss | The model's outputs (predictions) are compared to the ground truth and evaluated to see how wrong they are. | `loss = loss_fn(y_pred, y_test)`|
|3 | Calulate evaluation metrics (optional) | Alongside the loss value you may want to calculate other evaluation metrics such as accuracy on the test set. | Custom functions|

In [0]:
loss_fn = nn.L1Loss() # Mean Absolute Error (MAE)
optimizer = torch.optim.SGD(params=model_0.parameters(), lr=0.01) # stocastic gradient descent

In [0]:
torch.manual_seed(42)
# how many times the model will pass over the training data
epochs = 180

# create empty lists to track values
train_loss_values = []
test_loss_values = []
epoch_count = []

for epoch in range(epochs):
  # set the model to training mode
  model_0.train()

  # 1. Forward pass on train data using the forward() method inside
  y_pred = model_0(x_train)

  # 2. Calculate the loss (how different are our models predictions to the ground truth)
  loss = loss_fn(y_pred, y_train)

  # 3. Zero grad of the optimizer
  optimizer.zero_grad()

  # 4. Loss backwards
  loss.backward()

  # 5. Progress the optimizer
  optimizer.step()

  # testing

  # set the model to evaluation mode for testing
  model_0.eval()

  with torch.inference_mode():
    # 1. Do the forward pass
    test_pred = model_0(x_test)

    # 2. Calculate the loss
    test_loss = loss_fn(test_pred, y_test.type(torch.float)) # predictions come in torch.float datatype

    # Print out what's happening
    if epoch % 10 == 0:
        epoch_count.append(epoch)
        train_loss_values.append(loss.detach().numpy())
        test_loss_values.append(test_loss.detach().numpy())
        print(f"Epoch: {epoch} | MAE Train Loss: {loss} | MAE Test Loss: {test_loss} ")

In [0]:
# Plot the loss curves
plt.plot(epoch_count, train_loss_values, label="Train loss")
plt.plot(epoch_count, test_loss_values, label="Test loss")
plt.title("Training and test loss curves")
plt.ylabel("Loss")
plt.xlabel("Epochs")
plt.legend()

In [0]:
# Find our model's learned parameters
print("The model learned the following values for weights and bias:")
print(model_0.state_dict())
print("\nAnd the original values for weights and bias are:")
print(f"weights: {weight}, bias: {bias}")

# Making predictions (inference)

Once you've trained a model, you'll likely want to make predictions with it.

We've already seen a glimpse of this in the training and testing code above, the steps to do it outside of the training/testing loop are similar.

There are three things to remember when making predictions (also called performing inference) with a PyTorch model:

1. Set the model in evaluation mode `model.eval()`.
2. Make the predictions using the inference mode context manager `with torch.inference_mode(): ...`
3. All predictions should be made with objects on the same device (e.g. data and model on GPU only or data and model on CPU only).


In [0]:
# 1. Set the model in evaluation mode
model_0.eval()

# 2. Setup the inference mode context manager
with torch.inference_mode():
  # 3. Make sure the calculations are done with the model and data on the same device
  # in our case, we haven't setup device-agnostic code yet so our data and model are
  # on the CPU by default.
  model_0.to(device)
  x_test = x_test.to(device)
  y_preds = model_0(x_test)

y_preds

In [0]:
plot_predictions(
    train_data=x_train,
    train_labels=y_train,
    test_data=x_test,
    test_labels=y_test,
    predictions=y_preds.cpu()
)

# Saving a model

In [0]:
from pathlib import Path

# 1. Create models directory 
model_path = Path("models")
model_path.mkdir(parents=True, exist_ok=True)

# 2. Create model save path 
model_name = "intro_linear_regressions_model_0.pth"
model_path = model_path / model_name

# 3. Save the model state dict 
print(f"Saving model to: {Path.absolute(model_save_path)}")
torch.save(
  obj=model_0.state_dict(), # only saving the state_dict() only saves the models learned parameters
  f=model_path
) 

# Loading a model

In [0]:


# Instantiate a new instance of our model (this will be instantiated with random weights)
loaded_model_0 = LinearRegressionModel()

# Load the state_dict of our saved model (this will update the new instance of our model with trained weights)
loaded_model_0.load_state_dict(torch.load(f=model_path))


In [0]:
#make predictions
# 1. Put the loaded model into evaluation mode
loaded_model_0.eval()

# 2. Use the inference mode context manager to make predictions
with torch.inference_mode():
    loaded_model_preds = loaded_model_0(x_test) # perform a forward pass on the test data with the loaded model

# Compare previous model predictions with loaded model predictions (these should be the same)
y_preds == loaded_model_preds
